In [1]:
!pip install -q -U google-genai faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.5/262.5 kB 18.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.59.0 which is incompatible.


Importing

In [2]:
import numpy as np
import faiss

from google import genai
from google.genai import types
from google.colab import userdata

connecting to gemini

In [3]:
client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

EMBEDDING_MODEL = "gemini-embedding-001"
EMBEDDING_DIMENSION = 768

MODEL = "gemini-3.1-flash-lite"

creating embedding function

In [4]:
def embed(text):

    response = client.models.embed_content(
        model=EMBEDDING_MODEL,
        contents=text,
        config=types.EmbedContentConfig(
            output_dimensionality=EMBEDDING_DIMENSION
        )
    )

    return response.embeddings[0].values

FAQ information

In [5]:
faqs = [
    {
        "question": "How do I place an order?",
        "answer": "Open the app, select a restaurant, choose your food items, add them to your cart, select your delivery address and payment method, and click Place Order."
    },

    {
        "question": "How can I track my order?",
        "answer": "Open My Orders and select your current order to check the order status and delivery information."
    },

    {
        "question": "Can I cancel my order?",
        "answer": "You can cancel your order if the restaurant has not started preparing it."
    },

    {
        "question": "My order is delayed. What should I do?",
        "answer": "Check the estimated delivery time in My Orders. If the order is significantly delayed, contact customer support through the Help section."
    },

    {
        "question": "I received the wrong food. What should I do?",
        "answer": "Open My Orders, select the order and go to Help. Report the incorrect item."
    },

    {
        "question": "How can I request a refund?",
        "answer": "Open the relevant order, go to Help and submit a refund request if the order qualifies."
    },

    {
        "question": "How do I contact customer support?",
        "answer": "Open the app and go to the Help or Support section."
    },

    {
        "question": "How do I reset my password?",
        "answer": "On the login screen, select Forgot Password and follow the instructions to reset your password."
    }
]

FAQ text

In [6]:
faq_texts = [
    faq["question"] + " " + faq["answer"]
    for faq in faqs
]

print("Number of FAQs:", len(faq_texts))

Number of FAQs: 8


FAQ into numbers

In [7]:
faq_embeddings = np.array([
    embed(text)
    for text in faq_texts
])

print("FAQ embedding shape:", faq_embeddings.shape)

FAQ embedding shape: (8, 768)


normalize

In [8]:
faq_embeddings = faq_embeddings / np.linalg.norm(
    faq_embeddings,
    axis=1,
    keepdims=True
)

print("Normalized FAQ vectors:", faq_embeddings.shape)

Normalized FAQ vectors: (8, 768)


FAISS index

In [9]:
dimension = faq_embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)

index.add(
    faq_embeddings.astype("float32")
)

print("===== FAQ INDEX =====")
print("Index built successfully.")
print("Number of FAQs:", index.ntotal)
print("Vector dimension:", dimension)

===== FAQ INDEX =====
Index built successfully.
Number of FAQs: 8
Vector dimension: 768


search the FAQ

In [10]:
def faq_search(query, top_k=3):

    q_vec = np.array(
        embed(query),
        dtype="float32"
    )

    q_vec = (
        q_vec / np.linalg.norm(q_vec)
    ).reshape(1, -1)

    scores, indices = index.search(
        q_vec,
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        results.append({
            "question": faqs[idx]["question"],
            "answer": faqs[idx]["answer"],
            "score": float(score)
        })

    return results

test the search

In [11]:
results = faq_search(
    "My food delivery is taking too long"
)

for result in results:

    print(
        f"{result['score']:.3f} → "
        f"{result['question']}"
    )

0.720 → My order is delayed. What should I do?
0.649 → Can I cancel my order?
0.643 → I received the wrong food. What should I do?


giving to gemini

In [12]:
def answer_question(user_question):

    results = faq_search(user_question, top_k=3)

    context = "\n\n".join([
        f"Question: {r['question']}\n"
        f"Answer: {r['answer']}"
        for r in results
    ])

    prompt = f"""
You are a FoodExpress customer support assistant.

Use ONLY the FAQ information below.

Do not make up information.

If the information is not available, say:
"Sorry, I don't have this information."

FAQ INFORMATION:

{context}

CUSTOMER QUESTION:

{user_question}

Give a short and simple answer.
"""

    response = client.models.generate_content(
        model=MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.2,
            max_output_tokens=200
        )
    )

    return response.text

make it a chatbot

In [ ]:
print("===== FoodExpress RAG Bot =====")
print("Type 'bye' to exit.\n")

while True:

    user_question = input("You: ")

    if user_question.lower() in ["bye", "quit"]:
        print("Bot: Thank you!")
        break

    answer = answer_question(user_question)

    print("Bot:", answer)
    print()

===== FoodExpress RAG Bot =====
Type 'bye' to exit.

You: My order is delayed
Bot: Check the estimated delivery time in My Orders. If the order is significantly delayed, contact customer support through the Help section.

You: How do I reset my password
Bot: On the login screen, select Forgot Password and follow the instructions to reset your password.

You: I received the wrong food. What should I do?
Bot: Open My Orders, select the order and go to Help. Report the incorrect item.

